In [1]:
# ===== Step 9.1: PlantDoc detection boxes -> matched to our images =====
import os, glob, shutil, subprocess, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mp
W = "/kaggle/working"; os.makedirs(f"{W}/feats", exist_ok=True); os.makedirs(f"{W}/cache", exist_ok=True)
ROOT = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/figures/table_kd_main.csv", recursive=True)[0]))
for f in ["pd_manifest.csv", "pw_manifest.csv"]: shutil.copy(f"{ROOT}/feats/{f}", f"{W}/feats/{f}")
shutil.copytree(f"{ROOT}/figures", f"{W}/figures", dirs_exist_ok=True)
if not os.path.exists(f"{W}/cache/pd_256.npy"): os.symlink(f"{ROOT}/cache/pd_256.npy", f"{W}/cache/pd_256.npy")
print("restored from", ROOT)

# 1) clone detection labels
REPO = f"{W}/plantdoc_det"
if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset", REPO],
                       capture_output=True, text=True); print(r.stdout[-300:], r.stderr[-300:])
csvs = [c for c in glob.glob(f"{REPO}/**/*.csv", recursive=True)]
print("label csvs:", [os.path.relpath(c, REPO) for c in csvs])
L = pd.concat([pd.read_csv(c).assign(split=os.path.basename(c)) for c in csvs], ignore_index=True)
print("columns:", list(L.columns)); print(L.head(3).to_string())

# 2) match by file name
PD = pd.read_csv(f"{W}/feats/pd_manifest.csv", low_memory=False)
norm = lambda s: re.sub(r"[^a-z0-9]", "", os.path.splitext(os.path.basename(str(s)).lower())[0])
PD["key"] = PD.orig_path.map(norm); L["key"] = L.filename.map(norm)
dupkeys = PD.key.duplicated(keep=False).sum(); print(f"\nPD images: {len(PD)} | ambiguous names in PD: {dupkeys}")
M = L.merge(PD[["uid", "key", "class_name", "orig_w", "orig_h", "p2_role", "in_clean"]], on="key", how="inner", suffixes=("_box", ""))
M = M[~M.key.isin(PD.key[PD.key.duplicated(keep=False)])]                     # drop ambiguous names
print(f"box rows: {len(L)} | matched rows: {len(M)} | matched images: {M.uid.nunique()} / {len(PD)}")
if "width" in M and "height" in M:
    size_ok = (np.abs(M.width - M.orig_w) <= 2) & (np.abs(M.height - M.orig_h) <= 2)
    print(f"image size agrees: {size_ok.mean():.3f}")
    M = M[size_ok]
cls_col = "class" if "class" in M else [c for c in M.columns if "class" in c.lower() and c != "class_name"][0]
agree = (M[cls_col].str.lower().str.replace("_", " ") == M.class_name.str.lower().str.replace("_", " "))
print(f"box class == our image class: {agree.mean():.3f}")
print("  disagreement examples:", M.loc[~agree, [cls_col, "class_name"]].drop_duplicates().head(6).values.tolist())

# 3) boxes -> 256x256 crop coordinates (short side resized to 256, centre crop)
S = 256
def to_crop(r):
    w, h = r.orig_w, r.orig_h; s = S / min(w, h); nw, nh = max(S, round(w*s)), max(S, round(h*s))
    ox, oy = (nw - S) // 2, (nh - S) // 2
    x1, y1, x2, y2 = r.xmin * nw / w - ox, r.ymin * nh / h - oy, r.xmax * nw / w - ox, r.ymax * nh / h - oy
    return pd.Series(dict(x1=np.clip(x1, 0, S), y1=np.clip(y1, 0, S), x2=np.clip(x2, 0, S), y2=np.clip(y2, 0, S)))
B = pd.concat([M[["uid", cls_col, "split"]].reset_index(drop=True), M.apply(to_crop, axis=1).reset_index(drop=True)], axis=1)
B = B[(B.x2 - B.x1 > 2) & (B.y2 - B.y1 > 2)]
B.to_csv(f"{W}/feats/pd_boxes_256.csv", index=False)
cover = []
for u, g in B.groupby("uid"):
    m = np.zeros((S, S), bool)
    for _, r in g.iterrows(): m[int(r.y1):int(np.ceil(r.y2)), int(r.x1):int(np.ceil(r.x2))] = True
    cover.append(m.mean())
cl = PD.set_index("uid").loc[B.uid.unique()]
print(f"\nimages with boxes: {B.uid.nunique()} | in clean set: {int((cl.in_clean == True).sum())} | "
      f"boxes/image: {len(B)/B.uid.nunique():.2f} | box coverage median: {np.median(cover):.2f}")

# 4) visual check
ARR = np.load(f"{W}/cache/pd_256.npy", mmap_mode="r"); row_of = dict(zip(PD.uid, range(len(PD))))
pick = np.random.default_rng(0).choice(B.uid.unique(), 12, replace=False)
fig, axs = plt.subplots(2, 6, figsize=(15, 5.4))
for ax, u in zip(axs.ravel(), pick):
    ax.imshow(ARR[row_of[u]]); ax.axis("off"); ax.set_title(PD.class_name.values[row_of[u]][:24], fontsize=7)
    for _, r in B[B.uid == u].iterrows():
        ax.add_patch(mp.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, ec="yellow", lw=1.5))
fig.suptitle("PlantDoc detection boxes mapped onto our 256×256 images", fontsize=10)
for e in ["png", "pdf"]: fig.savefig(f"{W}/figures/figD15_pd_boxes_check.{e}", dpi=200, bbox_inches="tight")
plt.close(fig); print("saved figD15_pd_boxes_check\nStep 9.1 DONE.")

restored from /kaggle/input/datasets/mashfii/wildleaf-step7
 ting files:  93% (4836/5200)
Updating files:  94% (4888/5200)
Updating files:  95% (4940/5200)
Updating files:  96% (4992/5200)
Updating files:  97% (5044/5200)
Updating files:  98% (5096/5200)
Updating files:  99% (5148/5200)
Updating files: 100% (5200/5200)
Updating files: 100% (5200/5200), done.

label csvs: ['test_labels.csv', 'train_labels.csv']
columns: ['filename', 'width', 'height', 'class', 'xmin', 'ymin', 'xmax', 'ymax', 'split']
                                            filename  width  height                 class  xmin  ymin  xmax  ymax            split
0                     Black%20rot%20on%20foliage.jpg   1060     795  grape leaf black rot     1    58   859   732  test_labels.csv
1                                    pepper_leaf.jpg    300     400      Bell_pepper leaf    40    58   251   392  test_labels.csv
2  depositphotos_1323264-Raspberry-leaf-on-white.jpg   1024     772        Raspberry leaf    98    75 

In [2]:
# ===== Step 9.2: Grad-CAM localisation vs PlantDoc boxes (P3 test folds, baseline vs KD) =====
import os, glob, numpy as np, pandas as pd, torch, torch.nn.functional as F, timm, matplotlib.pyplot as plt, matplotlib.patches as mp
from scipy.stats import wilcoxon
W, FIG = "/kaggle/working", "/kaggle/working/figures"; dev = "cuda"
ROOT = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/figures/table_kd_main.csv", recursive=True)[0]))
PD = pd.read_csv(f"{W}/feats/pd_manifest.csv", low_memory=False); B = pd.read_csv(f"{W}/feats/pd_boxes_256.csv")
ARR = np.load(f"{W}/cache/pd_256.npy", mmap_mode="r"); row_of = dict(zip(PD.uid, range(len(PD)))); S = 256
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
masks = {}
for u, g in B.groupby("uid"):
    m = np.zeros((S, S), bool)
    for _, r in g.iterrows(): m[int(r.y1):int(np.ceil(r.y2)), int(r.x1):int(np.ceil(r.x2))] = True
    masks[u] = m
yy, xx = np.mgrid[0:S, 0:S]; centre = np.exp(-(((xx - S/2)**2 + (yy - S/2)**2) / (2 * (S/4)**2))); centre /= centre.sum()
MODELS = {"mobilenetv3_large_100": ("mobilenetv3_large_100.ra_in1k", "MobileNetV3-L"), "efficientnet_b0": ("efficientnet_b0.ra_in1k", "EfficientNet-B0")}

def gradcam(net, idx):
    x = torch.from_numpy(np.ascontiguousarray(ARR[idx])).to(dev).permute(0, 3, 1, 2).float() / 255.
    x = (F.interpolate(x, size=224, mode="bicubic", antialias=True, align_corners=False).clamp(0, 1) - MEAN) / STD
    feat = net.forward_features(x); feat.retain_grad()
    logits = net.forward_head(feat); pred = logits.argmax(1)
    logits.gather(1, pred[:, None]).sum().backward()
    w = feat.grad.mean((2, 3), keepdim=True); cam = F.relu((w * feat).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=S, mode="bilinear", align_corners=False)[:, 0]
    return cam.detach().cpu().numpy(), pred.cpu().numpy()

rows, examples = [], {}
for short, (tname, nice) in MODELS.items():
    for meth, suf in [("baseline", ""), ("kd", "_kd0.5t4")]:
        for k in range(5):
            ck = f"{ROOT}/students/ckpt/pd_p3_f{k}_{short}_s0_lr0.001{suf}.pt"
            if not os.path.exists(ck): print("missing", ck); continue
            net = timm.create_model(tname, pretrained=False, num_classes=int(PD.class_id.max()) + 1).to(dev)
            net.load_state_dict(torch.load(ck, map_location=dev)); net.eval()
            test_u = [u for u in PD.uid[PD[f"p3_f{k}_role"] == "test"] if u in masks]
            idx_all = np.array([row_of[u] for u in test_u])
            for i in range(0, len(idx_all), 32):
                idx = idx_all[i:i+32]; cams, preds = gradcam(net, idx)
                for j, r in enumerate(idx):
                    u = PD.uid.values[r]; m = masks[u]; c = cams[j]; e = c / c.sum() if c.sum() > 0 else np.full_like(c, 1 / c.size)
                    area = m.mean(); ein = e[m].sum()
                    rows.append(dict(model=nice, method=meth, fold=k, uid=u, correct=int(preds[j] == PD.class_id.values[r]),
                                     box_area=area, energy_in=ein, lift=ein / area, hit=bool(m.flat[c.argmax()]),
                                     centre_energy_in=centre[m].sum()))
                    if u not in examples: examples[u] = {}
                    examples[u][(nice, meth)] = c
            net.zero_grad(); del net; torch.cuda.empty_cache()
        print("done", nice, meth, flush=True)

G = pd.DataFrame(rows); G.to_csv(f"{FIG}/table_gradcam_per_image.csv", index=False)
def summary(df):
    return df.groupby(["model", "method"]).agg(images=("uid", "size"), energy_in=("energy_in", "mean"), lift=("lift", "mean"),
                                               pointing_hit=("hit", "mean"), box_area=("box_area", "mean"),
                                               centre_prior=("centre_energy_in", "mean")).round(4)
pd.set_option("display.width", 220)
Sall = summary(G); Ssmall = summary(G[G.box_area < 0.4])
print("\n=== GRAD-CAM LOCALISATION (all images with boxes) ===\n", Sall.to_string())
print("\n=== small-box subset (boxes < 40% of image) ===\n", Ssmall.to_string())
print("\n(uniform/chance energy-in-box = box_area; centre_prior = energy-in-box of a centred Gaussian blob)")
for nice in G.model.unique():
    P = G[G.model == nice].pivot_table(index="uid", columns="method", values="energy_in").dropna()
    Ps = G[(G.model == nice) & (G.box_area < 0.4)].pivot_table(index="uid", columns="method", values="energy_in").dropna()
    print(f"{nice}: KD - baseline energy-in-box = {100*(P.kd - P.baseline).mean():+.2f} pp (n={len(P)}, Wilcoxon p={wilcoxon(P.kd, P.baseline).pvalue:.2e}) | "
          f"small boxes: {100*(Ps.kd - Ps.baseline).mean():+.2f} pp (n={len(Ps)}, p={wilcoxon(Ps.kd, Ps.baseline).pvalue:.2e})")
pd.concat([Sall.assign(subset="all"), Ssmall.assign(subset="box<40%")]).to_csv(f"{FIG}/table_gradcam_localisation.csv")

# fig9_1: examples (small-box images, both methods correct for MobileNetV3-L)
cand = G[(G.model == "MobileNetV3-L") & (G.box_area < 0.4)].pivot_table(index="uid", columns="method", values="correct").dropna()
cand = cand[(cand.baseline == 1) & (cand.kd == 1)].index.tolist()
pick = list(np.random.default_rng(3).choice(cand, min(6, len(cand)), replace=False))
fig, axs = plt.subplots(3, len(pick), figsize=(2.3 * len(pick), 7.2))
for c, u in enumerate(pick):
    im = ARR[row_of[u]]
    axs[0, c].imshow(im); axs[0, c].set_title(PD.class_name.values[row_of[u]][:22], fontsize=7)
    for _, r in B[B.uid == u].iterrows():
        axs[0, c].add_patch(mp.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, ec="yellow", lw=1.2))
    for rr, meth in [(1, "baseline"), (2, "kd")]:
        cam = examples[u][("MobileNetV3-L", meth)]; cam = cam / (cam.max() + 1e-8)
        axs[rr, c].imshow(im); axs[rr, c].imshow(cam, cmap="jet", alpha=0.45)
        ein = G[(G.uid == u) & (G.model == "MobileNetV3-L") & (G.method == meth)].energy_in.iloc[0]
        axs[rr, c].set_title(f"{meth}: {ein:.0%} in box", fontsize=7)
for a in axs.ravel(): a.axis("off")
axs[1, 0].text(-20, 128, "Baseline", rotation=90, va="center", fontsize=9); axs[2, 0].text(-20, 128, "KD", rotation=90, va="center", fontsize=9)
fig.suptitle("Grad-CAM of MobileNetV3-L on unseen PlantDoc test images (yellow = annotated diseased leaves)", fontsize=10)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig9_1_gradcam_examples.{e}", dpi=250, bbox_inches="tight")
plt.close(fig)

# fig9_2: localisation bars with chance and centre-prior references
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, (Sx, title) in zip(axs, [(Sall, "All images with boxes"), (Ssmall, "Small boxes (< 40% of image)")]):
    idx = Sx.index.tolist(); x = np.arange(len(idx))
    ax.bar(x, Sx.energy_in, color=["#8C8C8C" if m == "baseline" else "#4C72B0" for _, m in idx])
    for i, v in enumerate(Sx.energy_in): ax.text(i, v + .01, f"{v:.2f}", ha="center", fontsize=8)
    ax.axhline(Sx.box_area.mean(), color="k", ls=":", label="chance (uniform)")
    ax.axhline(Sx.centre_prior.mean(), color="#C44E52", ls="--", label="centre prior")
    ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{mt}" for m, mt in idx], fontsize=7); ax.set_title(title, fontsize=9)
axs[0].set_ylabel("Grad-CAM energy inside leaf boxes"); axs[0].legend(fontsize=7)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig9_2_gradcam_localisation.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG)
log = log[~log.file.isin(["fig9_1_gradcam_examples", "fig9_2_gradcam_localisation", "figD15_pd_boxes_check"])]
pd.concat([log, pd.DataFrame([
    dict(file="figD15_pd_boxes_check", step="9", caption="PlantDoc detection boxes mapped onto our 256×256 images (visual check)."),
    dict(file="fig9_1_gradcam_examples", step="9", caption="Grad-CAM of baseline vs KD MobileNetV3-L on unseen PlantDoc test images; yellow boxes = annotated diseased leaves."),
    dict(file="fig9_2_gradcam_localisation", step="9", caption="Share of Grad-CAM energy inside annotated leaf boxes, vs chance and a centre prior.")])]).to_csv(LOG, index=False)
print("saved fig9_1_gradcam_examples, fig9_2_gradcam_localisation\nStep 9.2 DONE.")

done MobileNetV3-L baseline
done MobileNetV3-L kd
done EfficientNet-B0 baseline
done EfficientNet-B0 kd

=== GRAD-CAM LOCALISATION (all images with boxes) ===
                           images  energy_in    lift  pointing_hit  box_area  centre_prior
model           method                                                                   
EfficientNet-B0 baseline    2286     0.7028  1.2639        0.8193    0.5884         0.698
                kd          2286     0.7172  1.2939        0.8430    0.5884         0.698
MobileNetV3-L   baseline    2286     0.6703  1.1885        0.7857    0.5884         0.698
                kd          2286     0.6786  1.2148        0.8132    0.5884         0.698

=== small-box subset (boxes < 40% of image) ===
                           images  energy_in    lift  pointing_hit  box_area  centre_prior
model           method                                                                   
EfficientNet-B0 baseline     561     0.3723  1.4735        0.5526    0

In [3]:
import os
from IPython.display import FileLink
os.chdir("/kaggle/working")
os.system("rm -f step9.zip && zip -qr step9.zip figures feats/pd_boxes_256.csv")
print(round(os.path.getsize("step9.zip")/1e6), "MB"); FileLink("step9.zip")

56 MB


/kaggle/working/step9.zip